In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


from src.core import file_manager as fm
from src.nlu_builder.scripts import setup_data

In [3]:
actor = 'patient'
variation = 'without_others_intent/k100_without_sentences_higher_than_median'
work_dir = fm.filename_from_data_dir( f'output/{actor}/{variation}')

sentences_to_ignore = pd.read_csv(f'{work_dir}/intersection_300_sentences.csv')

In [21]:
file = fm.filename_from_data_dir( f'output/patient/{variation}/intersection_300_sentences_with_label.csv')

data_to_valid = pd.read_csv(file)

data_to_valid = data_to_valid.rename(columns={"intent": "manual"})


In [30]:
from src.embeddings.constants import  EMBEDDING_MODELS_TRANSLATION

df_merged = data_to_valid

for embedding in EMBEDDING_MODELS_TRANSLATION.keys():
  df_embedding = pd.read_csv(f'{work_dir}/{embedding}/annotated_sentences.csv')
  df_embedding = df_embedding.drop(['annotated_txt', 'label', 'distance'], axis=1).rename(columns={"intent": embedding})

  df_merged = pd.merge(df_merged, df_embedding, on='txt', how='left')

df_merged

Merging embedding: bert_pt
Merging embedding: flair_pt
Merging embedding: glove
Merging embedding: lasbe
Merging embedding: use


,txt,manual,bert_pt,flair_pt,glove,lasbe,use
0,Não faço uso de remédio. Estou tomando devido ...,inform_medicine,inform_medicine,inform_symptoms,inform_symptoms,inform_medicine,inform_medicine
1,O que vc me aconselha?,request_inform,request_inform,request_inform,inform_symptoms,request_inform,request_inform
2,Não sinto gosto e nem cheiro,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms
3,Dor de cabeça e dor garganta,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms
4,Só hoje estou sentindo dificuldade pra respirar,inform_symptoms,inform_symptoms,inform_symptoms,inform_medicine,inform_symptoms,inform_symptoms
...,...,...,...,...,...,...,...
295,Dor no corpo já tive febre estou com diarreia ...,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms
296,"Boa tarde, olha eu nao tive mais febre porém h...",inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms
297,E só sinto falta de ar qdo me dá crise de tosse,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms
298,Só dor no corpo e na cabeça,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms,inform_symptoms


In [49]:
manual_labels = df['manual'].value_counts().to_dict()
manual_labels

{'inform_symptoms': 182,
 'greeting': 58,
 'request_inform': 43,
 'inform_medicine': 14,
 'others': 3}

In [54]:
for embedding in EMBEDDING_MODELS_TRANSLATION.keys():
  predictions = df[df['manual'] == df[embedding]]['manual'].value_counts().to_dict()
  
  print(f'Predições para o {embedding}')
  for intent in predictions:
    total = manual_labels[intent]
    correct_predictions = predictions[intent]
    percentage = correct_predictions / total

    print(f'{intent}, corretas: {correct_predictions} of {total}, percentage: {percentage}')
  print('=================================\n')

Predições para o bert_pt
inform_symptoms, corretas: 176 of 182, percentage: 0.967032967032967
greeting, corretas: 57 of 58, percentage: 0.9827586206896551
request_inform, corretas: 34 of 43, percentage: 0.7906976744186046
inform_medicine, corretas: 5 of 14, percentage: 0.35714285714285715

Predições para o flair_pt
inform_symptoms, corretas: 181 of 182, percentage: 0.9945054945054945
greeting, corretas: 58 of 58, percentage: 1.0
request_inform, corretas: 32 of 43, percentage: 0.7441860465116279
inform_medicine, corretas: 12 of 14, percentage: 0.8571428571428571

Predições para o glove
inform_symptoms, corretas: 179 of 182, percentage: 0.9835164835164835
greeting, corretas: 55 of 58, percentage: 0.9482758620689655
request_inform, corretas: 14 of 43, percentage: 0.32558139534883723
inform_medicine, corretas: 8 of 14, percentage: 0.5714285714285714

Predições para o lasbe
inform_symptoms, corretas: 182 of 182, percentage: 1.0
greeting, corretas: 58 of 58, percentage: 1.0
request_inform, c

In [4]:
file = fm.filename_from_data_dir( f'output/patient/{variation}/intersection_300_sentences_with_label.csv')

data_to_valid = pd.read_csv(file)

data_to_valid = data_to_valid.loc[data_to_valid.intent != 'others']

data_to_valid.intent.value_counts()

setup_data.generate_nlu_file_from_df(data_to_valid, f'{work_dir}/intersection_300_sentences_with_label.yml')

generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/intersection_300_sentences_with_label.yml
getting intents...
The intent: greeting, has 58 examples
The intent: inform_medicine, has 14 examples
The intent: inform_symptoms, has 182 examples
The intent: request_inform, has 43 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/intersection_300_sentences_with_label.yml


In [3]:
def create_train_test_data(embedding_name):
  embedding_dir = f'{work_dir}/{embedding_name}'

  df = fm.read_annotated_df_with_embeddings(embedding_name)

  df_to_use = df.loc[~df['txt'].isin(sentences_to_ignore['txt'])]
  df_train, df_test = train_test_split(df_to_use, test_size=0.3, random_state=42)

  df_train.to_csv(f'{embedding_dir}/training_data.csv', index=False)
  df_test.to_csv(f'{embedding_dir}/test_data.csv', index=False)

  setup_data.generate_nlu_file_from_df(df_train, f'{embedding_dir}/training_data.yml')
  setup_data.generate_nlu_file_from_df(df_test, f'{embedding_dir}/test_data.yml')
  
  return df_train.txt.shape, df_test.txt.shape

In [5]:
create_train_test_data('bert_pt')

generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/bert_pt/training_data.yml
getting intents...
The intent: greeting, has 785 examples
The intent: inform_medicine, has 623 examples
The intent: inform_symptoms, has 3699 examples
The intent: request_inform, has 868 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/bert_pt/training_data.yml
generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/bert_pt/test_data.yml
getting intents...
The intent: greeting, has 332 examples
The intent: inform_medicine, has 255 examples
The intent: inform_symptoms, has 1599 examples
The intent: request_inform, has 376 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_inten

((5975,), (2562,))

In [6]:
create_train_test_data('flair_pt')

generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/flair_pt/training_data.yml
getting intents...
The intent: greeting, has 685 examples
The intent: inform_medicine, has 293 examples
The intent: inform_symptoms, has 4232 examples
The intent: request_inform, has 768 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/flair_pt/training_data.yml
generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/flair_pt/test_data.yml
getting intents...
The intent: greeting, has 297 examples
The intent: inform_medicine, has 123 examples
The intent: inform_symptoms, has 1800 examples
The intent: request_inform, has 342 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_in

((5978,), (2562,))

In [7]:
create_train_test_data('glove')

generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/glove/training_data.yml
getting intents...
The intent: greeting, has 621 examples
The intent: inform_medicine, has 230 examples
The intent: inform_symptoms, has 4839 examples
The intent: request_inform, has 330 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/glove/training_data.yml
generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/glove/test_data.yml
getting intents...
The intent: greeting, has 272 examples
The intent: inform_medicine, has 100 examples
The intent: inform_symptoms, has 2062 examples
The intent: request_inform, has 146 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100

((6020,), (2580,))

In [8]:
create_train_test_data('lasbe')

generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/lasbe/training_data.yml
getting intents...
The intent: greeting, has 677 examples
The intent: inform_medicine, has 664 examples
The intent: inform_symptoms, has 3138 examples
The intent: request_inform, has 915 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/lasbe/training_data.yml
generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/lasbe/test_data.yml
getting intents...
The intent: greeting, has 277 examples
The intent: inform_medicine, has 317 examples
The intent: inform_symptoms, has 1351 examples
The intent: request_inform, has 368 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100

((5394,), (2313,))

In [9]:
create_train_test_data('use')

generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/use/training_data.yml
getting intents...
The intent: greeting, has 462 examples
The intent: inform_medicine, has 646 examples
The intent: inform_symptoms, has 2771 examples
The intent: request_inform, has 760 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/use/training_data.yml
generating /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_without_sentences_higher_than_median/use/test_data.yml
getting intents...
The intent: greeting, has 206 examples
The intent: inform_medicine, has 275 examples
The intent: inform_symptoms, has 1149 examples
The intent: request_inform, has 359 examples
the content was saved in: /home/valmir/dev/python/intent_classifier/data/output/patient/without_others_intent/k100_witho

((4639,), (1989,))